In [1]:
import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine
from datetime import date

#### Conexion con las bases de datos

In [ ]:
config = {
    'host': 'localhost',       
    'port': 
    'database': 'mensajeria',  
    'user': 'postgres',            
    'password': 
}

In [3]:
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

    config_co = config['SOURCE_DB']
    config_etl = config['ETL_PRO']

url_co = f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:{config_co['port']}/{config_co['dbname']}"
url_etl = f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:{config_etl['port']}/{config_etl['dbname']}"


engine_origen = create_engine(url_co)
engine_destino = create_engine(url_etl)

In [4]:
dim_novedad = pd.read_sql_table('dim_novedad', engine_destino, schema='data_mart_novedades')
dim_mensajero = pd.read_sql_table('dim_mensajero', engine_destino, schema='data_mart_novedades')
dim_tiempo = pd.read_sql_table('dim_tiempo', engine_destino, schema='data_mart_novedades')
dim_hora = pd.read_sql_table('dim_hora', engine_destino, schema='data_mart_novedades')

In [5]:
dim_novedad.head()

,id_tipo_novedad,tipo_novedad
0,2,No puedo continuar
1,1,Novedades del servicio


In [6]:
dim_mensajero.head()

,id_mensajero,activo
0,1,True
1,42,True
2,48,True
3,41,True
4,13,True


In [7]:
dim_tiempo.head()

,fecha,id_tiempo,año,mes,dia,dia_semana,fin_de_semana
0,2023-09-19,1,2023,9,19,1,0
1,2023-09-20,2,2023,9,20,2,0
2,2023-09-21,3,2023,9,21,3,0
3,2023-09-22,4,2023,9,22,4,0
4,2023-09-23,5,2023,9,23,5,1


In [8]:
dim_hora.head()

,id_hora,hora
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4


In [9]:
novedad = pd.read_sql_table('mensajeria_novedadesservicio', engine_origen)

In [10]:
hecho_novedad = pd.DataFrame()

In [11]:
hecho_novedad["id_novedad_servicio"] = novedad["id"]
hecho_novedad["id_tipo_novedad"] = novedad["tipo_novedad_id"]
hecho_novedad["id_mensajero"] = novedad["mensajero_id"]
hecho_novedad["id_hora"] = pd.to_datetime(novedad["fecha_novedad"]).dt.hour
hecho_novedad["fecha"] = pd.to_datetime(novedad["fecha_novedad"]).dt.tz_localize(None).dt.normalize()
dim_tiempo["fecha"] = pd.to_datetime(dim_tiempo["fecha"]).dt.tz_localize(None)

In [12]:
hecho_novedad = hecho_novedad.merge(
    dim_tiempo[["id_tiempo", "fecha"]],
    on="fecha",
    how="left"
)

hecho_novedad.drop(columns=["fecha"], inplace=True)

In [13]:
hecho_novedad["cod_servicio"] = novedad["servicio_id"]
hecho_novedad["cantidad_novedades"] = 1

In [14]:
columnas_finales = [
    "id_novedad_servicio",
    "id_tipo_novedad",
    "id_mensajero",
    "id_tiempo",
    "id_hora",
    "cod_servicio",
    "cantidad_novedades",
]

hecho_novedad = hecho_novedad[columnas_finales]

In [15]:
hecho_novedad.head()

,id_novedad_servicio,id_tipo_novedad,id_mensajero,id_tiempo,id_hora,cod_servicio,cantidad_novedades
0,4,1,7,73,5,51,1
1,5,1,7,73,5,51,1
2,6,1,7,73,5,51,1
3,7,1,7,73,5,51,1
4,8,1,7,73,5,51,1
